Programa para la obtención de la velocidad de giro del sistema $\Omega$ mediante un ajuste por mínimos cuadrado de los datos experimentales de la sueprficie libre del fluido proporcionados por Tracker a una parábola de la forma $z=ax^2+bx+c.\
Este programa proporciona la representación del ajuste realizado junto con los datos experimentales, los parámetros $a$, $b$, $c$ que producen un mujer ajuste, bondad de ajuste y el valor de la velocidad de giro calculada en rad/s y en rpm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#################################################################################################################
#######################################Inserte aquí el fichero obtenido de Tracker###############################
#################################################################################################################
# Cambia este nombre para ajustar otro archivo del directorio.
fichero = "1_parabola.txt"




G = 980  # Aceleración gravitatoria en cm/s²


# Función para convertir cadenas con coma decimal a float
def _to_float(s):
    if isinstance(s, bytes):
        s = s.decode()
    return float(s.replace(",", "."))

# Cargar datos desde el archivo, omitiendo las dos primeras filas y usando solo las columnas 1 y 2
data = np.loadtxt(fichero,skiprows=2,usecols=(1, 2),converters={1: _to_float, 2: _to_float})
x = data[:, 0]
z = data[:, 1]

# Función para ajustar una parábola completa (con término lineal) y calcular covarianza

def parabola(x, z):
    """Ajusta z = a x^2 + b x + c por mínimos cuadrados y devuelve covarianza."""
    A = np.column_stack([x**2, x, np.ones_like(x)])
    # parms=(a,b,c) del ajuste, se obtiene por mínimos cuadrados con np.linalg.lstsq
    params, _, _, _ = np.linalg.lstsq(A, z, rcond=None)

    n = len(z) # numeros de datos
    p = 3 # número de parámetros (a, b, c)
    residuals = z - np.dot(A, params)
    rss = np.sum(residuals**2)
    #grados de libertad
    grados_de_libertad = n - p

    # Ahora calculamos la covarianza de los parámtros a,b,c

    s2 = rss / grados_de_libertad
    cov = s2 * np.linalg.inv(np.dot(A.T, A))
    

    return params, cov


(a, b, c), cov =parabola(x, z)
err_a, err_b, err_c = np.sqrt(np.diag(cov))

# Error vertical constante obtenido imponiendo chi^2 reducido = 1
residuals = z - (a * x**2 + b * x + c)
grados_de_libertad = len(z) - 3
sigma_y = np.sqrt(np.sum(residuals**2) / grados_de_libertad)
chi2_red = np.sum((residuals / sigma_y) ** 2) / grados_de_libertad

# RMSE y residuo máximo
rmse = np.sqrt(np.mean(residuals**2))
max_resid = np.max(np.abs(residuals))

# Calcular omega a partir del parámetro a
# a = ω²/(2g)  →  ω = √(2g·a)
omega = np.sqrt(2 * G * a)
err_omega = omega * err_a / (2 * a)  # Propagación de error

print(f"Archivo: {fichero}")
print(f"N puntos: {len(x)}")
print(f"\nParámetros del ajuste z(x) = a·x² + b·x + c:")
print(f"a = {a:.6g} ± {err_a:.2g} cm⁻¹")
print(f"b = {b:.6g} ± {err_b:.2g} (adimensional)")
print(f"c = {c:.6g} ± {err_c:.2g} cm")
print(f"\nError vertical obtenido con χ² reducido = 1:")
print(f"σ_y = {sigma_y:.4f} cm")
print(f"χ² reducido = {chi2_red:.3f}")
print()
print("Valores estadísticos del ajuste:")
print(f"RMSE = {rmse:.4f} cm")
print(f"Residuo máximo absoluto = {max_resid:.4f} cm")
print(f"\nVelocidad angular (del líquido en rotación):")
print(f"ω = {omega:.4f} ± {err_omega:.4f} rad/s")
print(f"ω = {omega*60/(2*np.pi):.2f} ± {err_omega*60/(2*np.pi):.2f} rpm")


x_d = np.linspace(np.min(x), np.max(x), 400)
z_d = a * x_d**2 + b * x_d + c

plt.figure(figsize=(10, 6))
plt.errorbar(
    x,
    z,
    yerr=sigma_y,
    fmt="o",
    markersize=4,
    color="tab:blue",
    ecolor="tab:blue",
    elinewidth=1,
    capsize=3,
    label="Datos experimentales", 
)
plt.plot(x_d, z_d, label=r"Ajuste $z(x)=ax^2+bx+c$", color="tab:red", lw=2)
plt.xlabel("x (cm)", fontsize=11)
plt.ylabel("z (cm)", fontsize=11)
plt.grid(alpha=0.3)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()